# Human-in-the-Loop Approval Workflows

## Purpose
Learn how to require human approval before agents execute sensitive operations. This is critical for production systems where actions like refunds, data deletion, or financial transactions need explicit authorization.

## Key Concepts
- **needs_approval**: Flag that pauses execution for human review
- **result.interruptions**: List of pending approval requests
- **state.approve()**: Grant permission for tool execution
- **state.reject()**: Deny permission and return rejection message
- **Resume Pattern**: Run → Check interruptions → Approve/Reject → Resume

## Installation

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

## Authentication Setup

In [ ]:
model_id = "openai.gpt-5.5"

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)  # OpenAI-platform tracing can't reach Mantle

## Import Libraries

In [ ]:
import asyncio
from agents import Agent, Runner, function_tool

## Part 1: Agent-as-Tool Approval

Convert a specialized agent into a tool that requires human approval.

**Pattern**: Agent → Convert to tool with `needs_approval=True` → Use in orchestrator

💡 **Use Case**: When an entire agent's capabilities need approval (e.g., billing agent, data deletion agent).

### Step 1: Create Specialist Agent

Define an agent that handles refunds:

In [ ]:
refund_agent = Agent(
    name="Refund agent",
    model=model_id,
    instructions=(
        "You process customer refunds. Given an order, state clearly that "
        "the refund has been issued and for how much."
    ),
)

### Step 2: Convert to Approval-Required Tool

Use `as_tool()` with `needs_approval=True`:

⚡ **Key**: Execution pauses before this agent runs, waiting for human approval.

In [ ]:
refund_tool = refund_agent.as_tool(
    tool_name="process_refund",
    tool_description="Process a customer refund. Requires human approval.",
    needs_approval=True,  # Requires approval before execution
)

### Step 3: Create Orchestrator Agent

Build main agent that uses the approval-required tool:

In [ ]:
orchestrator = Agent(
    name="Support orchestrator",
    model=model_id,
    instructions=(
        "You handle support requests. If the customer asks for a refund, "
        "use the process_refund tool. Otherwise, answer directly."
    ),
    tools=[refund_tool],
)

### Step 4: Define Approval Helper

Create function to prompt for human approval:

In [ ]:
def prompt_approval(tool_name: str, arguments: str | None) -> bool:
    """Helper function to prompt for approval."""
    answer = input(f"Approve `{tool_name}` with args {arguments}? [y/N]: ").strip().lower()
    return answer in {"y", "yes"}

### Step 5: Implement Approval Loop

Run agent and handle approval workflow:

**Approval Flow**:
1. Agent decides to call approval-required tool
2. Execution pauses, `result.interruptions` contains pending approvals
3. Convert result to state: `state = result.to_state()`
4. For each interruption: approve (`state.approve()`) or reject (`state.reject()`)
5. Resume with updated state: `await Runner.run(orchestrator, state)`
6. Repeat until no more interruptions

🔍 **Watch**: Execution pauses at the tool call, waits for your input!

In [ ]:
result = await Runner.run(
    orchestrator,
    "Please refund order #12345 for $50.",
)

# Handle approval loop
while result.interruptions:
    state = result.to_state()

    for interruption in result.interruptions:
        approved = prompt_approval(
            interruption.name or "unknown_tool",
            interruption.arguments,
        )
        if approved:
            state.approve(interruption)  # Approve and continue
        else:
            state.reject(interruption)   # Reject and return rejection text

    # Resume with the updated state
    result = await Runner.run(orchestrator, state)

print("\nFinal output:\n", result.final_output)

## Part 2: Function Tool Approval

Mark individual function tools as requiring approval.

**Pattern**: `@function_tool(needs_approval=True)` → Same approval loop

💡 **Use Case**: When specific functions need approval (e.g., delete_user, transfer_funds).

### Step 6: Create Approval-Required Function

Use decorator parameter to require approval:

⚡ **Key**: Function body only executes AFTER approval is granted!

In [ ]:
@function_tool(needs_approval=True)  # Requires approval
def process_refund(order_id: str, amount: float) -> str:
    """Refund a given amount to a customer order."""
    # This body only runs AFTER approval
    return f"Refunded ${amount:.2f} for order {order_id}."

### Step 7: Create Agent with Approval Tool

In [ ]:
agent = Agent(
    name="Refund agent",
    instructions="Help the customer. Use process_refund to issue refunds.",
    tools=[process_refund],
    model=model_id,
)

### Step 8: Run with Approval Loop

Same approval pattern as agent-as-tool:

🎯 **Result**: Function only executes after you approve!

In [ ]:
result = await Runner.run(agent, "Refund $49.99 for order A-1001.")

while result.interruptions:
    state = result.to_state()
    for interruption in result.interruptions:
        tool = interruption.raw_item.name
        args = interruption.raw_item.arguments
        print(f"\n⚠️  Approval needed: {tool}({args})")

        answer = input("Approve? [y/n]: ").strip().lower()
        if answer == "y":
            state.approve(interruption)
        else:
            state.reject(interruption)

    # Resume with the decisions applied
    result = await Runner.run(agent, state)

print("\nFinal output:\n", result.final_output)

## 🎉 Congratulations!

You've completed the **Human-in-the-Loop Approval** notebook and the entire **Advanced Tools** section!